In [1]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive"
DRIVE_DIR = ROOT + "/files"
DATA_DIR = DRIVE_DIR + "/data"

for name in ("sources.yaml", "fetch_raw.py"):
    assert os.path.exists(f"{DRIVE_DIR}/{name}"), f"{name} absent de {DRIVE_DIR}"
    shutil.copy(f"{DRIVE_DIR}/{name}", ".")
print("Scripts copies depuis", DRIVE_DIR, "| donnees dans", DATA_DIR)

Mounted at /content/drive
Scripts copies depuis /content/drive/MyDrive/files | donnees dans /content/drive/MyDrive/files/data


In [ ]:
import zipfile
import requests
import csv
import os

DATA_DIR_REAL = "/content/drive/MyDrive/files/data"
EXPERIMENTS = {
    "zea_mays": "E-GEOD-50191",
    "oryza_sativa": "E-MTAB-2037",
}

def download_experiment_zip(accession, dest_dir):
    url = f"https://www.ebi.ac.uk/gxa/experiments/download/zip?accession={accession}"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    zpath = f"{dest_dir}/{accession}.zip"
    with open(zpath, "wb") as f:
        f.write(resp.content)
    with zipfile.ZipFile(zpath) as z:
        names = z.namelist()
        z.extractall(dest_dir)
    return names

# 1. inspecter dim_protein.csv une fois, pour confirmer les noms de colonnes
dim_protein_path = f"{DATA_DIR_REAL}/normalized/dim_protein.csv"
with open(dim_protein_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("colonnes de dim_protein.csv :", reader.fieldnames)
    sample_rows = [next(reader) for _ in range(3)]
    print("exemple de lignes :")
    for r in sample_rows:
        print(" ", r)

def load_plaza_gene_ids(species):
    ids = set()
    with open(dim_protein_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("assemblage") == species and row.get("plaza_gene_id"):
                ids.add(row["plaza_gene_id"])
    return ids

for species, accession in EXPERIMENTS.items():
    print(f"\n=== {species} — {accession} ===")
    dest = f"{DATA_DIR_REAL}/raw_expression_atlas_{species}"
    os.makedirs(dest, exist_ok=True)
    try:
        files = download_experiment_zip(accession, dest)
    except Exception as e:
        print(f"  echec telechargement : {e}")
        continue
    print(f"  fichiers recus : {files}")

    tpm_file = next((f for f in files if "tpms" in f.lower()), None)
    if not tpm_file:
        print("  AUCUN fichier TPM detecte -- a inspecter manuellement")
        continue

    ea_ids = set()
    with open(f"{dest}/{tpm_file}", encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or line.startswith("GeneID"):
                continue
            gid = line.split("\t")[0].strip()
            if gid:
                ea_ids.add(gid)

    plaza_ids = load_plaza_gene_ids(species)
    overlap = ea_ids & plaza_ids
    print(f"  identifiants Expression Atlas : {len(ea_ids)}")
    print(f"  identifiants plaza_gene_id (dim_protein, {species}) : {len(plaza_ids)}")
    print(f"  recoupement direct : {len(overlap)} ({100*len(overlap)/max(1,len(ea_ids)):.1f}% des IDs Expression Atlas)")
    print(f"  exemples EA non pontes : {list(ea_ids - plaza_ids)[:5]}")

colonnes de dim_protein.csv : ['assemblage', 'protein_accession', 'plaza_gene_id', 'chr', 'start', 'stop', 'strand']
exemple de lignes :
  {'assemblage': 'zea_mays', 'protein_accession': 'NP_001143585.1', 'plaza_gene_id': 'Zm00001eb000170', 'chr': '1', 'start': '552260', 'stop': '564691', 'strand': '-'}
  {'assemblage': 'zea_mays', 'protein_accession': 'NP_001146998.1', 'plaza_gene_id': 'Zm00001eb000200', 'chr': '1', 'start': '680019', 'stop': '680498', 'strand': '+'}
  {'assemblage': 'zea_mays', 'protein_accession': 'NP_001136827.1', 'plaza_gene_id': 'Zm00001eb000220', 'chr': '1', 'start': '1018420', 'stop': '1020406', 'strand': '+'}

=== zea_mays — E-GEOD-50191 ===
  fichiers recus : ['E-GEOD-50191/E-GEOD-50191-configuration.xml', 'E-GEOD-50191/E-GEOD-50191.condensed-sdrf.tsv', 'E-GEOD-50191/E-GEOD-50191.idf.txt', 'E-GEOD-50191/E-GEOD-50191-factors.xml', 'E-GEOD-50191/E-GEOD-50191-tpms.tsv']
  identifiants Expression Atlas : 44303
  identifiants plaza_gene_id (dim_protein, zea_mays) 

In [ ]:
#télécharger qtaro_sjis.zip, le décoder en Shift-JIS, et en inspecter la structure avant de normaliser :
import zipfile
import requests
import os

QTARO_URL = "https://dbarchive.biosciencedbc.jp/data/qtaro/LATEST/qtaro_sjis.zip"
dest_dir = f"{DATA_DIR}/raw_qtaro"
os.makedirs(dest_dir, exist_ok=True)

resp = requests.get(QTARO_URL, timeout=60)
resp.raise_for_status()
zpath = f"{dest_dir}/qtaro_sjis.zip"
with open(zpath, "wb") as f:
    f.write(resp.content)
print(f"telecharge : {len(resp.content)} octets")

with zipfile.ZipFile(zpath) as z:
    names = z.namelist()
    print("fichiers dans l'archive :", names)
    z.extractall(dest_dir)

# on tente une lecture en Shift-JIS (shift_jis / cp932) sur chaque fichier texte/csv/tsv trouve
for name in names:
    path = f"{dest_dir}/{name}"
    if not os.path.isfile(path):
        continue
    print(f"\n=== {name} ===")
    try:
        with open(path, encoding="shift_jis", errors="replace") as f:
            lines = [next(f) for _ in range(8)]
        for l in lines:
            print("  ", l.rstrip()[:200])
    except Exception as e:
        print(f"  echec lecture shift_jis : {e}")
        # fallback : afficher les octets bruts des premieres lignes pour diagnostiquer
        with open(path, "rb") as f:
            raw = f.read(300)
        print("  octets bruts (apercu) :", raw)

telecharge : 90649 octets
fichiers dans l'archive : ['qtaro_sjis.csv']

=== qtaro_sjis.csv ===
   id,QTL/Gene,Major category,Category of object character,Character,Marker,No. of marker for position determination,Chr,Genome start,Genome end,Mapping method,Population,No. of plants,LOD,Parent A,Paren
   1,Ph5-1,Morphological trait,Dwarf,plant height,RFLP,2,5,2069338,2760413,C)Interval,DHL,123,,IR64,Azucena,,,,,,RG313,RZ556,,,,,1998,pha,4,"Yan, J., Zhu, J., He, C., Benmoussa, M., and Wu, P. (1998). Mo
   2,hr6,Physiological trait,Eating quality,Head rice,SSR,2,6,1764586,5425602,C)Interval,DH from BC3F1,312,11.9,Caiapo,Oryza glaberrima (accession IRGC No. 103544),,,,,,RM190,RM253,,,54.1,-2.41,2004,pha,
   3,rb2,Physiological trait,Eating quality,Rice bran,SSR,2,2,2105542,2882252,C)Interval,DH from BC3F1,312,55.6,Caiapo,Oryza glaberrima (accession IRGC No. 103544),,,,,,RM236,RM279,,,32.7,-18.25,2004,pha
   4,alk6-1,Physiological trait,Eating quality,Alkali-spread.score,SSR,2,6,1764586,542560

In [ ]:
with open(f"{dest_dir}/qtaro_sjis.csv", encoding="shift_jis", errors="replace") as f:
    header = next(f).rstrip("\n").split(",")

print(f"{len(header)} colonnes :")
for i, h in enumerate(header):
    print(f"  [{i}] {h}")

# et une ligne complete en exemple, pour voir le contenu de chaque colonne
with open(f"{dest_dir}/qtaro_sjis.csv", encoding="shift_jis", errors="replace") as f:
    next(f)
    row = next(f).rstrip("\n").split(",")
print("\nexemple de ligne (id=1) :")
for h, v in zip(header, row):
    print(f"  {h} = {v}")

32 colonnes :
  [0] id
  [1] QTL/Gene
  [2] Major category
  [3] Category of object character
  [4] Character
  [5] Marker
  [6] No. of marker for position determination
  [7] Chr
  [8] Genome start
  [9] Genome end
  [10] Mapping method
  [11] Population
  [12] No. of plants
  [13] LOD
  [14] Parent A
  [15] Parent B
  [16] Direction (Parent)
  [17] a) Physical
  [18] b) Fine1; interval
  [19] b) Fine2; interval
  [20] b) Fine3; co-segregated
  [21] c) Interval1; interval
  [22] c) Interval2; interval
  [23] c) Interval3; co-segregated
  [24] d) Co-segregated
  [25] Explained variance
  [26] Additive effect
  [27] Year
  [28] Reference source
  [29] Reference no.
  [30] Reference
  [31] Reference location

exemple de ligne (id=1) :
  id = 1
  QTL/Gene = Ph5-1
  Major category = Morphological trait
  Category of object character = Dwarf
  Character = plant height
  Marker = RFLP
  No. of marker for position determination = 2
  Chr = 5
  Genome start = 2069338
  Genome end = 2760413
  M

In [ ]:
import csv

qtaro_path = f"{dest_dir}/qtaro_sjis.csv"

# 1. parsing propre (gere les champs entre guillemets avec virgules, ex. Reference)
qtl_rows = []
with open(qtaro_path, encoding="shift_jis", errors="replace", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        qtl_rows.append(row)

print(f"lignes totales dans qtaro_sjis.csv : {len(qtl_rows)}")

# 2. QC : combien ont des coordonnees physiques utilisables ?
usable = [r for r in qtl_rows if r.get("Chr") and r.get("Genome start") and r.get("Genome end")]
print(f"lignes avec Chr + Genome start + Genome end renseignes : {len(usable)} ({100*len(usable)/len(qtl_rows):.1f}%)")

# ecriture de fact_qtl.csv (table QTL nettoyee, independante de la jointure)
fact_qtl_path = f"{DATA_DIR}/normalized/fact_qtl.csv"
fieldnames = ["qtl_id","qtl_gene_name","major_category","trait_category","trait_detail",
              "chr","genome_start","genome_end","mapping_method","explained_variance",
              "additive_effect","year","reference"]
with open(fact_qtl_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in usable:
        writer.writerow({
            "qtl_id": r["id"],
            "qtl_gene_name": r.get("QTL/Gene", ""),
            "major_category": r.get("Major category", ""),
            "trait_category": r.get("Category of object character", ""),
            "trait_detail": r.get("Character", ""),
            "chr": r["Chr"],
            "genome_start": r["Genome start"],
            "genome_end": r["Genome end"],
            "mapping_method": r.get("Mapping method", ""),
            "explained_variance": r.get("Explained variance", ""),
            "additive_effect": r.get("Additive effect", ""),
            "year": r.get("Year", ""),
            "reference": r.get("Reference", ""),
        })
print(f"{len(usable)} lignes ecrites dans {fact_qtl_path}")

# Helper function to normalize chromosome identifiers (e.g., 'chr01' -> '1', 'chr12' -> '12', '5' -> '5')
def clean_chr(c_str):
    if not c_str:
        return ""
    c_clean = c_str.lower().replace("chr", "").strip()
    try:
        return str(int(c_clean)) # removes leading zeros
    except ValueError:
        return c_clean

# 3. jointure par chevauchement de coordonnees avec les genes riz de dim_protein.csv
rice_genes = []
with open(f"{DATA_DIR}/normalized/dim_protein.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["assemblage"] == "oryza_sativa" and row.get("chr") and row.get("start") and row.get("stop"):
            rice_genes.append(row)

print(f"\ngenes riz avec coordonnees (dim_protein) : {len(rice_genes)}")

from collections import defaultdict
genes_by_chr = defaultdict(list)
for g in rice_genes:
    norm_c = clean_chr(g["chr"])
    if norm_c:
        genes_by_chr[norm_c].append(g)

links = []
for qtl in usable:
    chr_ = clean_chr(qtl["Chr"])
    try:
        qstart, qend = int(qtl["Genome start"]), int(qtl["Genome end"])
    except ValueError:
        continue
    for g in genes_by_chr.get(chr_, []):
        gstart, gend = int(g["start"]), int(g["stop"])
        if gstart <= qend and gend >= qstart:  # chevauchement
            links.append({"qtl_id": qtl["id"], "protein_accession": g["protein_accession"],
                          "chr": qtl["Chr"], "qtl_start": qstart, "qtl_end": qend,
                          "gene_start": gstart, "gene_end": gend})

print(f"liens QTL-gene trouves (chevauchement de coordonnees) : {len(links)}")
qtl_with_gene = len({l["qtl_id"] for l in links})
print(f"QTL avec au moins un gene chevauchant : {qtl_with_gene}/{len(usable)} ({100*qtl_with_gene/len(usable):.1f}%)")

link_path = f"{DATA_DIR}/normalized/fact_qtl_gene_link.csv"
with open(link_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["qtl_id","protein_accession","chr","qtl_start","qtl_end","gene_start","gene_end"])
    writer.writeheader()
    writer.writerows(links)
print(f"ecrit : {link_path}")

lignes totales dans qtaro_sjis.csv : 1051
lignes avec Chr + Genome start + Genome end renseignes : 1051 (100.0%)
1051 lignes ecrites dans /content/drive/MyDrive/files/data/normalized/fact_qtl.csv

genes riz avec coordonnees (dim_protein) : 41610
liens QTL-gene trouves (chevauchement de coordonnees) : 340470
QTL avec au moins un gene chevauchant : 868/1051 (82.6%)
ecrit : /content/drive/MyDrive/files/data/normalized/fact_qtl_gene_link.csv


In [ ]:
import pandas as pd
N = f"{DATA_DIR}/normalized"
q = pd.read_csv(f"{N}/fact_qtl.csv")
l = pd.read_csv(f"{N}/fact_qtl_gene_link.csv")
print(q.columns.tolist())
print(l.columns.tolist())
q["interval_bp"] = q["genome_end"] - q["genome_start"] + 1
print(q.groupby("mapping_method")["interval_bp"].describe().round(0))
print(q["chr"].value_counts().sort_index())

['qtl_id', 'qtl_gene_name', 'major_category', 'trait_category', 'trait_detail', 'chr', 'genome_start', 'genome_end', 'mapping_method', 'explained_variance', 'additive_effect', 'year', 'reference']
['qtl_id', 'protein_accession', 'chr', 'qtl_start', 'qtl_end', 'gene_start', 'gene_end']
                 count       mean        std    min       25%        50%  \
mapping_method                                                             
A)Physical        50.0    79607.0    61285.0  748.0    8175.0    95480.0   
B)Fine            96.0  2100305.0  2915535.0  112.0  310519.0  1062174.0   
C)Interval       785.0  3567894.0  4754658.0 -485.0  707673.0  1965659.0   
d)Co-segregated  120.0     1762.0     2365.0 -626.0     185.0      480.0   

                       75%         max  
mapping_method                          
A)Physical        132664.0    162866.0  
B)Fine           2307866.0  13482398.0  
C)Interval       4353872.0  33886656.0  
d)Co-segregated     2808.0     12108.0  
chr
1     1

In [ ]:
import pandas as pd
N = f"{DATA_DIR}/normalized"
q = pd.read_csv(f"{N}/fact_qtl.csv")
d = pd.read_csv(f"{N}/dim_protein.csv", dtype=str)
d = d[d.assemblage.str.contains("IRGSP", na=False)].copy()
d["gene_id"] = d.plaza_gene_id.fillna(d.protein_accession)
d["chr"] = d.chr.astype(str).str.replace("Chr", "", regex=False)
d["start"] = d.start.astype(int)
d["stop"] = d.stop.astype(int)
g = d.groupby(["gene_id", "chr"]).agg(start=("start", "min"), stop=("stop", "max")).reset_index()
print("genes distincts :", len(g))

q["lo"] = q[["genome_start", "genome_end"]].min(axis=1)
q["hi"] = q[["genome_start", "genome_end"]].max(axis=1)
q["chr"] = q.chr.astype(str)
q["interval_bp"] = q.hi - q.lo + 1
q["resolution"] = pd.cut(q.interval_bp, [0, 500000, 2000000, 10**10], labels=["fine", "medium", "coarse"]).astype(str)

m = q[["qtl_id", "chr", "lo", "hi"]].merge(g, on="chr")
m = m[(m.start <= m.hi) & (m.stop >= m.lo)]
link = m[["qtl_id", "gene_id", "chr", "lo", "hi", "start", "stop"]].rename(
    columns={"lo": "qtl_start", "hi": "qtl_end", "start": "gene_start", "stop": "gene_end"})
n = link.groupby("qtl_id").gene_id.nunique().rename("n_genes_in_interval")
q = q.merge(n, on="qtl_id", how="left")
q["n_genes_in_interval"] = q.n_genes_in_interval.fillna(0).astype(int)

print("liens gene-QTL :", len(link), "| QTL avec gene :", (q.n_genes_in_interval > 0).sum(), "/", len(q))
print(q.groupby("resolution").n_genes_in_interval.describe().round(1))
print(link[(link.chr == "10") & (link.qtl_id == 6)].sort_values("gene_start").head(10))

q.to_csv(f"{N}/fact_qtl.csv", index=False)
link.to_csv(f"{N}/fact_qtl_gene_link.csv", index=False)

genes distincts : 0
liens gene-QTL : 0 | QTL avec gene : 0 / 1051
            count  mean  std  min  25%  50%  75%  max
resolution                                           
coarse      420.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
fine        379.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
medium      252.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
Empty DataFrame
Columns: [qtl_id, gene_id, chr, qtl_start, qtl_end, gene_start, gene_end]
Index: []


In [ ]:
import pandas as pd
d = pd.read_csv(f"{DATA_DIR}/normalized/dim_protein.csv", dtype=str)
print(d.assemblage.value_counts(dropna=False))
print(d.groupby("assemblage").chr.unique().apply(lambda x: list(x)[:6]))
print(d.groupby("assemblage").plaza_gene_id.apply(lambda s: s.notna().mean()).round(3))

assemblage
nicotiana_tabacum_k326    64776
nicotiana_tabacum_tn90    60627
oryza_sativa              41610
zea_mays                  23594
Name: count, dtype: int64
assemblage
nicotiana_tabacum_k326                                         [nan]
nicotiana_tabacum_tn90                                         [nan]
oryza_sativa              [chr01, chr02, chr03, chr04, chr05, chr06]
zea_mays                     [1, scaf_494, 9, 4, scaf_640, scaf_612]
Name: chr, dtype: object
assemblage
nicotiana_tabacum_k326    0.0
nicotiana_tabacum_tn90    0.0
oryza_sativa              1.0
zea_mays                  1.0
Name: plaza_gene_id, dtype: float64


In [ ]:
import pandas as pd
N = f"{DATA_DIR}/normalized"
q = pd.read_csv(f"{N}/fact_qtl.csv")
d = pd.read_csv(f"{N}/dim_protein.csv", dtype=str)
d = d[d.assemblage == "oryza_sativa"].copy()
assert len(d) > 0, "aucune ligne riz"

d["chr"] = d.chr.str.extract(r"(\d+)")[0]
print("chr non reconnus :", d.chr.isna().sum(), "| valeurs :", sorted(d.chr.dropna().astype(int).unique()))
d = d.dropna(subset=["chr", "start", "stop"])
d["chr"] = d.chr.astype(int)
d["start"] = d.start.astype(int)
d["stop"] = d.stop.astype(int)
d["gene_id"] = d.plaza_gene_id.fillna(d.protein_accession)
g = d.groupby(["gene_id", "chr"]).agg(start=("start", "min"), stop=("stop", "max")).reset_index()
print("genes distincts :", len(g))

q["chr"] = q.chr.astype(int)
q["lo"] = q[["genome_start", "genome_end"]].min(axis=1)
q["hi"] = q[["genome_start", "genome_end"]].max(axis=1)
q["interval_bp"] = q.hi - q.lo + 1
q["resolution"] = pd.cut(q.interval_bp, [0, 500000, 2000000, 10**10], labels=["fine", "medium", "coarse"]).astype(str)

m = q[["qtl_id", "chr", "lo", "hi"]].merge(g, on="chr")
m = m[(m.start <= m.hi) & (m.stop >= m.lo)]
link = m[["qtl_id", "gene_id", "chr", "lo", "hi", "start", "stop"]].rename(
    columns={"lo": "qtl_start", "hi": "qtl_end", "start": "gene_start", "stop": "gene_end"})
n = link.groupby("qtl_id").gene_id.nunique().rename("n_genes_in_interval")
q = q.drop(columns=["n_genes_in_interval"], errors="ignore").merge(n, on="qtl_id", how="left")
q["n_genes_in_interval"] = q.n_genes_in_interval.fillna(0).astype(int)

print("liens :", len(link), "| QTL avec gene :", (q.n_genes_in_interval > 0).sum(), "/", len(q))
print(q.groupby("resolution").n_genes_in_interval.describe().round(1))
print(link[link.qtl_id == 6].sort_values("gene_start").head(10))

ok = len(g) > 20000 and 0 < len(link) < 340470 and (q.n_genes_in_interval > 0).sum() > 800
print("controles OK :", ok)
if ok:
    q.to_csv(f"{N}/fact_qtl.csv", index=False)
    link.to_csv(f"{N}/fact_qtl_gene_link.csv", index=False)
    print("fichiers ecrits")

chr non reconnus : 0 | valeurs : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
genes distincts : 26685
liens : 220155 | QTL avec gene : 868 / 1051
            count   mean    std   min    25%    50%    75%     max
resolution                                                        
coarse      420.0  448.6  323.7  30.0  249.8  360.5  582.0  2738.0
fine        379.0    8.3   15.5   0.0    0.0    1.0    8.5    70.0
medium      252.0  113.4   50.7   0.0   78.0  110.0  139.5   243.0
       qtl_id       gene_id  chr  qtl_start   qtl_end  gene_start  gene_end
12582       6  Os10g0471350   10   17481862  17627660    17475729  17484035
12583       6  Os10g0471400   10   17481862  17627660    17486423  17490000
12584       6  Os10g0471700   10   17481862  17627660    17500159  17503058
12585       6  Os10g0472400   10   17481862  17627660    17545507  17547580
12586       6  Os10g0472

In [ ]:
import pandas as pd
N = f"{DATA_DIR}/normalized"
d = pd.read_csv(f"{N}/dim_protein.csv", dtype=str)
d = d[d.assemblage == "oryza_sativa"]
test = ["Os10g0463400", "Os06g0275000", "Os01g0883800", "Os03g0407400", "Os01g0197700"]
print(d[d.plaza_gene_id.isin(test)][["plaza_gene_id", "chr", "start", "stop"]].drop_duplicates("plaza_gene_id"))
q = pd.read_csv(f"{N}/fact_qtl.csv")
print(q[q.qtl_gene_name.isin(["Ehd1", "Hd1", "sd1", "GS3", "Gn1a"])][["qtl_gene_name", "chr", "lo", "hi", "mapping_method"]])

      plaza_gene_id    chr     start      stop
24542  Os01g0197700  chr01   5270835   5275522
28604  Os01g0883800  chr01  38382387  38385208
37002  Os03g0407400  chr03  16729715  16735077
47846  Os06g0275000  chr06   9336535   9338359
59211  Os10g0463400  chr10  17076098  17081344
    qtl_gene_name  chr        lo        hi mapping_method
5            Ehd1   10  17481862  17627660     A)Physical
72            GS3    3  17343910  17472917     A)Physical
279           sd1    1  36697294  42364623     C)Interval
435           sd1    1  40038301  40288357         B)Fine


In [ ]:
import pandas as pd
N = f"{DATA_DIR}/normalized"
q = pd.read_csv(f"{N}/fact_qtl.csv")
print(q.groupby(pd.cut(q.year, [1995,2000,2005,2010,2015,2020,2025]))["interval_bp"].agg(["count","mean"]).round(0))

              count       mean
year                          
(1995, 2000]    171  4103255.0
(2000, 2005]    593  2600151.0
(2005, 2010]    258  2742422.0
(2010, 2015]      0        NaN
(2015, 2020]      0        NaN
(2020, 2025]      0        NaN


/tmp/ipykernel_5020/3849244269.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(q.groupby(pd.cut(q.year, [1995,2000,2005,2010,2015,2020,2025]))["interval_bp"].agg(["count","mean"]).round(0))


In [2]:
import shutil
import os

# Définition des chemins
DATA_DIR_TO_ZIP = "/content/drive/MyDrive/files/data"
ZIP_OUTPUT_NAME = "/content/drive/MyDrive/files/data_compressed"

print(f"Début de la compression du dossier : {DATA_DIR_TO_ZIP}")
if os.path.exists(DATA_DIR_TO_ZIP):
    # shutil.make_archive utilise la compression par défaut (ZIP_DEFLATED)
    # pour créer un fichier intact et plus léger.
    output_path = shutil.make_archive(ZIP_OUTPUT_NAME, 'zip', DATA_DIR_TO_ZIP)
    print(f"Compression terminée avec succès !")
    print(f"L'archive est disponible ici : {output_path}")
    print(f"Taille du fichier : {os.path.getsize(output_path) / (1024*1024):.2f} MB")
else:
    print(f"Erreur : Le dossier {DATA_DIR_TO_ZIP} n'existe pas.")

Début de la compression du dossier : /content/drive/MyDrive/files/data
Compression terminée avec succès !
L'archive est disponible ici : /content/drive/MyDrive/files/data_compressed.zip
Taille du fichier : 2876.44 MB
